# Merri krismas

Scrivi il il codice e scopri chi è la persona a cui devi fare il regalo



In [1]:
# secret_santa_full.py
# This script handles:
# 1) Generating Secret Santa assignments with encryption
# 2) Exporting the assignments and master key to files
# 3) Allowing participants to decrypt their assignment using a short key

import random
import secrets
from cryptography.fernet import Fernet

# -----------------------------
# 1. Participant list
# -----------------------------
Gift_Giver_not_encrypted = [
    "Alice",
    "Bob",
    "Conan"
]

# -----------------------------
# 2. Generate derangement
# -----------------------------
def generate_derangement(names):
    while True:
        shuffled = names[:]
        random.shuffle(shuffled)
        if all(a != b for a, b in zip(names, shuffled)):
            return dict(zip(names, shuffled))

Gift_Assignment = generate_derangement(Gift_Giver_not_encrypted)

# -----------------------------
# 3. Generate short keys (3–4 chars)
# -----------------------------
Decryption_Keys = {name: secrets.token_hex(2) for name in Gift_Giver_not_encrypted}

# -----------------------------
# 4. Encrypt each receiver
# -----------------------------
master_key = Fernet.generate_key()
cipher = Fernet(master_key)

Gift_Receiver_encrypted = {}
for giver in Gift_Giver_not_encrypted:
    receiver = Gift_Assignment[giver]
    key = Decryption_Keys[giver]
    payload = f"{key}:{receiver}".encode()
    encrypted = cipher.encrypt(payload).decode()
    Gift_Receiver_encrypted[giver] = encrypted

# -----------------------------
# 5. Export assignments to files
# -----------------------------
with open("assignments.txt", "w") as f:
    for giver in Gift_Giver_not_encrypted:
        f.write(f"{giver},{Gift_Receiver_encrypted[giver]},{Decryption_Keys[giver]}\n")

with open("master_key.txt", "wb") as f:
    f.write(master_key)

print("Assignments and master key exported.")


Assignments and master key exported.


In [2]:

# -----------------------------
# 6. Participant decryption
# -----------------------------

user_key = input("\nEnter your short decryption key to reveal your giftee: ").strip()

matched_giver = None
for giver, (_, short_key) in [(g, (Gift_Receiver_encrypted[g], Decryption_Keys[g])) for g in Gift_Giver_not_encrypted]:
    if short_key == user_key:
        matched_giver = giver
        break

if matched_giver is None:
    print("No assignment found. Wrong key.")
    exit()

encrypted_payload = Gift_Receiver_encrypted[matched_giver]

# Decrypt

decrypted = cipher.decrypt(encrypted_payload.encode()).decode()
key_in_file, receiver = decrypted.split(":")

if key_in_file != user_key:
    print("Key mismatch — possible tampering.")
else:
    print(f"You give a gift to: {receiver}")


You give a gift to: Bob


NameError: name 'Gift_Giver_not_encrypted' is not defined